# Product Review Summarization with Flan-T5

This notebook reads the sentiment-analyzed product reviews and generates summaries for each product using Google's Flan-T5 model.

## 1. Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings('ignore')

/mnt/c/Users/Georg/ownCloud/Dokumente/Ironhack/.venv_acr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the CSV file
df = pd.read_csv('data/product_reviews_sentiment_and_confidence_SVM_updated.csv')

# Display basic information
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nNumber of unique products: {df['product_id'].nunique()}")
df.head()

Dataset shape: (34627, 8)

Columns: ['product_id', 'reviews.text', 'rating', 'original_sentiment_from_rating', 'predicted_sentiment_SVM', 'prediction_confidence', 'cluster', 'name']

Number of unique products: 39


,product_id,reviews.text,rating,original_sentiment_from_rating,predicted_sentiment_SVM,prediction_confidence,cluster,name
0,AVqkIhwDv8e3D1O-lebb,This product so far has not disappointed. My c...,5.0,positive,positive,0.938887,4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."
1,AVqkIhwDv8e3D1O-lebb,great for beginner or experienced person. Boug...,5.0,positive,positive,0.932079,4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."
2,AVqkIhwDv8e3D1O-lebb,Inexpensive tablet for him to use and learn on...,5.0,positive,positive,0.947257,4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."
3,AVqkIhwDv8e3D1O-lebb,I've had my Fire HD 8 two weeks now and I love...,4.0,positive,positive,0.991933,4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."
4,AVqkIhwDv8e3D1O-lebb,I bought this for my grand daughter when she c...,5.0,positive,positive,0.999997,4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."


## 2. Load Flan-T5 Model (GPU Accelerated)

We'll use the `google/flan-t5-large` model for better summarization quality. The model will automatically use GPU if available (CUDA).

In [3]:
import torch

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

# Load the larger Flan-T5 model and tokenizer
# Options: flan-t5-small (80M), flan-t5-base (250M), flan-t5-large (780M), flan-t5-xl (3B), flan-t5-xxl (11B)
model_name = "google/flan-t5-large"  # Using large model for better quality
print(f"\nLoading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move model to GPU
model = model.to(device)

print(f"✓ Model loaded successfully on {device}!")
print(f"Model parameters: ~780M")

Using device: cuda


GPU: NVIDIA GeForce RTX 2070 SUPER
CUDA Version: 12.1

Loading google/flan-t5-large...


2025-11-14 14:06:57.491961: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✓ Model loaded successfully on cuda!
Model parameters: ~780M


In [4]:
# Check GPU memory if available
if torch.cuda.is_available():
    print(f"\nGPU Memory:")
    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  Total: {total_memory:.2f} GB")


GPU Memory:
  Allocated: 3.06 GB
  Reserved: 3.06 GB
  Total: 8.00 GB


## 2.1 Apply LoRA Fine-Tuning

Use LoRA (Low-Rank Adaptation) to efficiently fine-tune the model for better review summarization while keeping memory usage low.

In [5]:
# Install PEFT library and upgrade accelerate for LoRA
%pip install -q peft accelerate>=0.26.0 --upgrade

Note: you may need to restart the kernel to use updated packages.


In [6]:
from peft import LoraConfig, get_peft_model, TaskType

# First, ensure the base model is in training mode
model.train()

# Configure improved LoRA parameters for better fine-tuning
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,  # Sequence-to-sequence task
    r=32,  # Increased rank from 20 to 32 for more capacity
    lora_alpha=64,  # Increased alpha (2x rank is standard)
    lora_dropout=0.05,  # Reduced dropout from 0.1 to 0.05 for better learning
    target_modules=["q", "v", "k", "o"],  # Added key and output projections for better coverage
    inference_mode=False,  # IMPORTANT: Training mode, not inference
    bias="none"  # Don't train bias terms
)

print("Applying improved LoRA configuration to the model...")
print(f"LoRA Config:")
print(f"  Rank (r): {lora_config.r} (increased for more capacity)")
print(f"  Alpha: {lora_config.lora_alpha} (2x rank)")
print(f"  Dropout: {lora_config.lora_dropout} (reduced for better learning)")
print(f"  Target modules: {lora_config.target_modules} (all attention projections)")
print(f"  Task type: SEQ_2_SEQ_LM")
print(f"  Inference mode: {lora_config.inference_mode} (training enabled)")

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Ensure model is in training mode after LoRA application
model.train()

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✓ LoRA applied successfully!")
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Total parameters: {total_params:,}")
print(f"Parameter efficiency: Training only {100 * trainable_params / total_params:.2f}% of parameters!")

# Verify gradients are enabled
sample_trainable = [p for p in model.parameters() if p.requires_grad]
print(f"✓ Verified: {len(sample_trainable)} parameter tensors require gradients")

# Enable gradient checkpointing to save memory
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()
    print(f"✓ Gradient checkpointing enabled for memory efficiency")

Applying improved LoRA configuration to the model...
LoRA Config:
  Rank (r): 32 (increased for more capacity)
  Alpha: 64 (2x rank)
  Dropout: 0.05 (reduced for better learning)
  Target modules: {'q', 'v', 'k', 'o'} (all attention projections)
  Task type: SEQ_2_SEQ_LM
  Inference mode: False (training enabled)

✓ LoRA applied successfully!
Trainable parameters: 18,874,368 (2.35%)
Total parameters: 802,024,448
Parameter efficiency: Training only 2.35% of parameters!
✓ Verified: 576 parameter tensors require gradients
✓ Gradient checkpointing enabled for memory efficiency

✓ LoRA applied successfully!
Trainable parameters: 18,874,368 (2.35%)
Total parameters: 802,024,448
Parameter efficiency: Training only 2.35% of parameters!
✓ Verified: 576 parameter tensors require gradients
✓ Gradient checkpointing enabled for memory efficiency


### Prepare Training Data for LoRA Fine-Tuning

Create training examples from the review data to fine-tune the model for concise 50-word summaries.

In [7]:
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.optim import AdamW

# Create improved training dataset that encourages true summarization
class ReviewSummaryDataset(Dataset):
    def __init__(self, reviews, tokenizer, max_input_length=512, max_target_length=80):
        self.reviews = reviews
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length
    
    def __len__(self):
        return len(self.reviews)
    
    def __getitem__(self, idx):
        review = self.reviews.iloc[idx]
        
        # Get the actual review text (truncated to fit context)
        review_text = review['combined_reviews'][:1500]
        
        # Create instruction-tuned prompt
        input_text = f"""Summarize the following product reviews in approximately 50 words. Focus on the key points customers mention: quality, usability, problems, and overall satisfaction.

Reviews: {review_text}

Summary:"""
        
        # Create synthetic target by extracting key phrases from actual reviews
        # This teaches the model to pull content from input, not memorize templates
        words = review_text.lower().split()
        sentiment = review['predicted_sentiment_SVM']
        rating = review['avg_rating']
        
        # Extract key sentiment words from actual reviews
        positive_indicators = [w for w in words if w in ['great', 'excellent', 'love', 'perfect', 'best', 'amazing', 'good', 'easy', 'works', 'highly', 'recommend']]
        negative_indicators = [w for w in words if w in ['bad', 'poor', 'waste', 'broken', 'terrible', 'worst', 'disappointed', 'problem', 'issue', 'difficult', 'cheap']]
        feature_words = [w for w in words if w in ['quality', 'price', 'battery', 'screen', 'sound', 'design', 'size', 'performance', 'durability', 'features', 'value']]
        
        # Build target based on what's actually in the reviews
        if sentiment == 'positive' and rating >= 4.5:
            sentiment_phrase = "Users praise" if positive_indicators else "Customers report positive experiences with"
        elif sentiment == 'positive':
            sentiment_phrase = "Customers generally appreciate" if positive_indicators else "Users find value in"
        else:
            sentiment_phrase = "Users express concerns about" if negative_indicators else "Mixed feedback on"
        
        # Mention specific features if found
        feature_mention = f" the {feature_words[0]}" if feature_words else " this product"
        
        # Create a more dynamic target that forces content extraction
        target_template = f"{sentiment_phrase}{feature_mention}. Rating: {rating:.1f}/5."
        
        # Tokenize
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        target_encoding = self.tokenizer(
            target_template,
            max_length=self.max_target_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Replace padding token id's in labels with -100 so they are ignored in loss
        labels = target_encoding['input_ids'].squeeze().clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': labels
        }

# Ensure product_reviews exists
if 'product_reviews' not in globals():
    print("product_reviews not found — building it now from df...")
    product_reviews = df.groupby('product_id').agg({
        'reviews.text': lambda x: ' '.join(x.astype(str)),
        'rating': 'mean',
        'predicted_sentiment_SVM': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
    }).reset_index()
    product_reviews.rename(columns={
        'reviews.text': 'combined_reviews',
        'rating': 'avg_rating'
    }, inplace=True)

# Prepare training data - use more samples for better training
train_dataset = ReviewSummaryDataset(
    product_reviews.head(30),  # Increased from 20 to 30 products
    tokenizer
)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Sample input shape: {train_dataset[0]['input_ids'].shape}")
print(f"Sample label shape: {train_dataset[0]['labels'].shape}")
print(f"\n✓ Training dataset prepared with content-extraction approach")

product_reviews not found — building it now from df...
Training dataset size: 30
Sample input shape: torch.Size([512])
Sample label shape: torch.Size([80])

✓ Training dataset prepared with content-extraction approach


### Fine-Tune Model with LoRA

Train the model for a few epochs to adapt it for concise 50-word review summaries.

In [8]:
from torch.optim import AdamW
from tqdm import tqdm
import math

# Ensure model is in training mode and parameters require grad
model.train()

# Disable cache for training (required with gradient checkpointing)
if hasattr(model, 'config'):
    try:
        model.config.use_cache = False
    except Exception:
        pass

# Optionally ensure inputs require grad for checkpointing-enabled models
if hasattr(model, 'enable_input_require_grads'):
    try:
        model.enable_input_require_grads()
    except Exception:
        pass

# Verify trainable parameters
trainable_params = [p for p in model.parameters() if p.requires_grad]
if len(trainable_params) == 0:
    raise RuntimeError("No trainable parameters found! LoRA may not be configured correctly.")

print(f"Found {len(trainable_params)} trainable parameter tensors")
print(f"Total trainable parameters: {sum(p.numel() for p in trainable_params):,}")

# Training configuration
num_epochs = 3
batch_size = 2
learning_rate = 3e-4
gradient_accumulation_steps = 4

# Create data loader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Optimizer - only LoRA parameters
optimizer = AdamW(trainable_params, lr=learning_rate)

# Training loop
print(f"\nStarting fine-tuning for {num_epochs} epochs...")
print(f"Batch size: {batch_size}, Learning rate: {learning_rate}")
print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
print(f"Effective batch size: {batch_size * gradient_accumulation_steps}")
print(f"Batches per epoch: {len(train_loader)}\n")

total_loss = 0
step = 0

for epoch in range(num_epochs):
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    optimizer.zero_grad()
    accumulated_batches = 0
    
    for batch_idx, batch in enumerate(progress_bar):
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss / gradient_accumulation_steps
        
        # Backward pass and gradient accumulation
        loss.backward()
        accumulated_batches += 1
        
        # Step optimizer after accumulating gradients OR at end of epoch
        if accumulated_batches >= gradient_accumulation_steps or (batch_idx + 1) == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()
            step += 1
            accumulated_batches = 0
        
        epoch_loss += loss.item() * gradient_accumulation_steps
        total_loss += loss.item() * gradient_accumulation_steps
        
        # Update progress bar
        progress_bar.set_postfix({'loss': f'{loss.item() * gradient_accumulation_steps:.4f}', 'steps': step})
    
    avg_epoch_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} completed. Average loss: {avg_epoch_loss:.4f}, Steps: {step}")
    
    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

avg_total_loss = total_loss / (num_epochs * len(train_loader))
expected_total_steps = math.ceil(len(train_loader) / gradient_accumulation_steps) * num_epochs
print(f"\n✓ Fine-tuning completed!")
print(f"Average training loss: {avg_total_loss:.4f}")
print(f"Total training steps: {step} (expected ≈ {expected_total_steps})")
print(f"Total batches processed: {num_epochs * len(train_loader)}")

Found 576 trainable parameter tensors
Total trainable parameters: 18,874,368

Starting fine-tuning for 3 epochs...
Batch size: 2, Learning rate: 0.0003
Gradient accumulation steps: 4
Effective batch size: 8
Batches per epoch: 15



Epoch 1/3:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 1/3: 100%|██████████| 15/15 [00:14<00:00,  1.04it/s, loss=2.8857, steps=4]



Epoch 1 completed. Average loss: 3.4891, Steps: 4


Epoch 2/3: 100%|██████████| 15/15 [00:13<00:00,  1.11it/s, loss=2.3377, steps=8]



Epoch 2 completed. Average loss: 2.7281, Steps: 8


Epoch 3/3: 100%|██████████| 15/15 [00:14<00:00,  1.06it/s, loss=1.1897, steps=12]

Epoch 3 completed. Average loss: 1.8787, Steps: 12

✓ Fine-tuning completed!
Average training loss: 2.6986
Total training steps: 12 (expected ≈ 12)
Total batches processed: 45


In [9]:
# Set model back to eval mode for inference
model.eval()

# Check GPU memory after training
if torch.cuda.is_available():
    print(f"\nGPU Memory After Training:")
    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")


GPU Memory After Training:
  Allocated: 3.32 GB
  Reserved: 3.38 GB


## 3. Prepare Reviews by Product

Group all reviews by product_id to create combined review text for summarization.

In [10]:
# Group reviews by product
product_reviews = df.groupby('product_id').agg({
    'reviews.text': lambda x: ' '.join(x.astype(str)),
    'rating': 'mean',
    'predicted_sentiment_SVM': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    'cluster': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    'name': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]  # Get product name
}).reset_index()

product_reviews.rename(columns={
    'reviews.text': 'combined_reviews',
    'rating': 'avg_rating'
}, inplace=True)

print(f"Number of products: {len(product_reviews)}")
product_reviews.head()

Number of products: 39


,product_id,combined_reviews,avg_rating,predicted_sentiment_SVM,cluster,name
0,AV1YE_muvKc47QAVgpwE,The Amazon Fire TV works better and faster tha...,4.707278,positive,3,Amazon Fire Tv
1,AV1YnR7wglJLPUi8IJmi,I got a great tablet from a great company with...,4.424731,positive,1,Echo (White)
2,AV1YnRtnglJLPUi8IJmV,I was looking for a reader that was light enou...,4.772355,positive,1,Amazon Kindle Paperwhite - eBook reader - 4 GB...
3,AVpe7AsMilAPnD_xQ78G,I initially had trouble deciding between the p...,4.666667,positive,4,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
4,AVpe9CMS1cnluZ0-aoC5,Finally received the Kindle Lighted Leather Co...,4.000000,positive,1,Amazon Kindle Lighted Leather Cover


## 4. Analyze Summaries by Sentiment and Cluster

Group summaries by sentiment and cluster to identify patterns.

In [11]:
# Choose source dataframe for analysis
if 'summaries_df' in globals():
    source_df = summaries_df.copy()
    source_name = 'summaries_df'
elif 'product_reviews_with_counts' in globals():
    source_df = product_reviews_with_counts.copy()
    source_name = 'product_reviews_with_counts'
elif 'product_reviews' in globals():
    source_df = product_reviews.copy()
    source_name = 'product_reviews'
else:
    raise RuntimeError("No dataset found for analysis. Run the cells that build 'summaries_df' first (generate summaries).")

# Determine a column to count as "product_count"
if 'name' in source_df.columns:
    count_col = 'name'
elif 'product_id' in source_df.columns:
    count_col = 'product_id'
else:
    count_col = None

# Group by sentiment
if 'predicted_sentiment_SVM' not in source_df.columns:
    raise RuntimeError(f"Column 'predicted_sentiment_SVM' not found in {source_name}.")

if count_col:
    sentiment_summary = (
        source_df.groupby('predicted_sentiment_SVM')
        .agg(product_count=(count_col, 'count'), avg_rating=('avg_rating', 'mean'))
        .sort_values('product_count', ascending=False)
    )
else:
    sentiment_summary = source_df.groupby('predicted_sentiment_SVM').size().to_frame('product_count')
    if 'avg_rating' in source_df.columns:
        sentiment_summary['avg_rating'] = source_df.groupby('predicted_sentiment_SVM')['avg_rating'].mean()

print("Products by Sentiment:")
print(sentiment_summary)
print("\n" + "="*80 + "\n")

# Group by cluster
if 'cluster' not in source_df.columns:
    raise RuntimeError(f"Column 'cluster' not found in {source_name}.")

if count_col:
    cluster_summary = (
        source_df.groupby('cluster')
        .agg(product_count=(count_col, 'count'), avg_rating=('avg_rating', 'mean'))
        .sort_values('product_count', ascending=False)
    )
else:
    cluster_summary = source_df.groupby('cluster').size().to_frame('product_count')
    if 'avg_rating' in source_df.columns:
        cluster_summary['avg_rating'] = source_df.groupby('cluster')['avg_rating'].mean()

print("Products by Cluster:")
print(cluster_summary)

Products by Sentiment:
                         product_count  avg_rating
predicted_sentiment_SVM                           
positive                            39    4.413732


Products by Cluster:
         product_count  avg_rating
cluster                           
4                   21    4.593766
1                   11    4.093692
3                    5    4.264108
0                    2    4.657649


## 5. Calculate Product Popularity Rankings

Create popularity rankings for products within each cluster based on average rating and review count.

In [12]:
# First, get review counts per product from original data
review_counts = df.groupby('product_id').agg({
    'reviews.text': 'count'
}).rename(columns={'reviews.text': 'review_count'}).reset_index()

# Merge review counts with product_reviews
product_reviews_with_counts = product_reviews.merge(review_counts, on='product_id', how='left')

# Calculate popularity score (weighted combination of rating and review count)
# Normalize both metrics to 0-1 scale
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
product_reviews_with_counts['rating_normalized'] = scaler.fit_transform(product_reviews_with_counts[['avg_rating']])
product_reviews_with_counts['count_normalized'] = scaler.fit_transform(product_reviews_with_counts[['review_count']])

# Popularity score: 70% rating, 30% review count
product_reviews_with_counts['popularity_score'] = (
    0.7 * product_reviews_with_counts['rating_normalized'] + 
    0.3 * product_reviews_with_counts['count_normalized']
)

# Rank products within each cluster
product_reviews_with_counts['popularity_rank_in_cluster'] = (
    product_reviews_with_counts.groupby('cluster')['popularity_score']
    .rank(ascending=False, method='dense')
    .astype(int)
)

# Add a 'popular' label based on ranking (top 3 in cluster = "High", 4-7 = "Medium", rest = "Low")
def categorize_popularity(rank, cluster_size):
    if rank <= 3:
        return 'High'
    elif rank <= 7:
        return 'Medium'
    else:
        return 'Low'

cluster_sizes = product_reviews_with_counts.groupby('cluster').size()
product_reviews_with_counts['cluster_size'] = product_reviews_with_counts['cluster'].map(cluster_sizes)
product_reviews_with_counts['popular'] = product_reviews_with_counts.apply(
    lambda row: categorize_popularity(row['popularity_rank_in_cluster'], row['cluster_size']), 
    axis=1
)

print("Product popularity rankings calculated!")
print(f"\nDistribution of popularity levels:")
print(product_reviews_with_counts['popular'].value_counts())
product_reviews_with_counts[['product_id', 'name', 'cluster', 'avg_rating', 'review_count', 
                              'popularity_score', 'popularity_rank_in_cluster', 'popular']].head(10)

Product popularity rankings calculated!

Distribution of popularity levels:
popular
Low       17
High      11
Medium    11
Name: count, dtype: int64


,product_id,name,cluster,avg_rating,review_count,popularity_score,popularity_rank_in_cluster,popular
0,AV1YE_muvKc47QAVgpwE,Amazon Fire Tv,3,4.707278,5056,0.757584,2,High
1,AV1YnR7wglJLPUi8IJmi,Echo (White),1,4.424731,372,0.551516,6,Medium
2,AV1YnRtnglJLPUi8IJmV,Amazon Kindle Paperwhite - eBook reader - 4 GB...,1,4.772355,3176,0.724093,1,High
3,AVpe7AsMilAPnD_xQ78G,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",4,4.666667,15,0.608464,9,Low
4,AVpe9CMS1cnluZ0-aoC5,Amazon Kindle Lighted Leather Cover,1,4.000000,5,0.424352,8,Low
5,AVpfIfGA1cnluZ0-emyp,Amazon,1,4.205479,73,0.482875,7,Medium
6,AVpf_4sUilAPnD_xlwYV,Amazon,1,3.066667,15,0.167252,10,Low
7,AVpf_znpilAPnD_xlvAF,Amazon Digital Services Inc.,1,3.500000,10,0.286610,9,Low
8,AVpff7_VilAPnD_xc1E_,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,4,5.000000,1,0.700000,4,Medium
9,AVpfiBlyLJeJML43-4Tp,Amazon,1,2.461538,13,0.000328,11,Low


## 6. View Top Products in Each Cluster

Display the most popular products in each cluster.

In [13]:
# Show top 3 products from each cluster
print("Top 3 Most Popular Products in Each Cluster:\n")
for cluster in sorted(product_reviews_with_counts['cluster'].unique()):
    cluster_data = product_reviews_with_counts[product_reviews_with_counts['cluster'] == cluster]
    top_in_cluster = cluster_data.nsmallest(3, 'popularity_rank_in_cluster')
    
    print(f"Cluster {cluster}:")
    for idx, row in top_in_cluster.iterrows():
        print(f"  Rank {int(row['popularity_rank_in_cluster'])}: {row['name']}")
        print(f"    Rating: {row['avg_rating']:.2f} | Reviews: {int(row['review_count'])} | Popularity: {row['popular']}")
        print(f"    Score: {row['popularity_score']:.3f}")
    print()

Top 3 Most Popular Products in Each Cluster:

Cluster 0:
  Rank 1: Amazon
    Rating: 4.88 | Reviews: 8 | Popularity: High
    Score: 0.666
  Rank 2: Amazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders
    Rating: 4.44 | Reviews: 401 | Popularity: High
    Score: 0.557

Cluster 1:
  Rank 1: Amazon Kindle Paperwhite - eBook reader - 4 GB - 6 monochrome Paperwhite - touchscreen - Wi-Fi - black
    Rating: 4.77 | Reviews: 3176 | Popularity: High
    Score: 0.724
  Rank 2: Kindle Voyage E-reader, 6 High-Resolution Display (300 ppi) with Adaptive Built-in Light, PagePress Sensors, Wi-Fi - Includes Special Offers,
    Rating: 4.73 | Reviews: 580 | Popularity: High
    Score: 0.641
  Rank 3: Echo (White)
    Rating: 4.70 | Reviews: 10 | Popularity: High
    Score: 0.618

Cluster 3:
  Rank 1: Echo (White)
    Rating: 4.67 | Reviews: 6619 | Popularity: High
    Score: 0.790
  Rank 2: Amazon Fire Tv
    Rating: 4.71 | Reviews: 5056 | Popularity: High
    Sco

## 7. Generate Product Summaries with Prompt Engineering

Create detailed summaries for each product using Flan-T5 with few-shot examples to guide the model.

## 7.1 Text Preprocessing for Better Summarization

Clean and preprocess review text to improve summary quality by removing noise, fixing formatting issues, and standardizing the text.

In [14]:
import re
import string

def clean_review_text(text):
    """
    Clean and preprocess review text for better summarization.
    
    Args:
        text: Raw review text
    
    Returns:
        Cleaned text
    """
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase for consistency
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s.,!?-]', ' ', text)
    
    # Fix multiple spaces
    text = re.sub(r'\s+', ' ', text)
    
    # Remove very short words (likely typos or fragments)
    words = text.split()
    words = [w for w in words if len(w) > 1 or w in ['a', 'i']]
    
    # Rejoin and trim
    text = ' '.join(words).strip()
    
    # Limit length to avoid overwhelming the model
    max_chars = 3000
    if len(text) > max_chars:
        # Try to cut at sentence boundary
        text = text[:max_chars]
        last_period = text.rfind('.')
        if last_period > max_chars * 0.7:  # Only cut at period if it's reasonably far in
            text = text[:last_period + 1]
    
    return text

In [15]:
def create_few_shot_prompt(reviews_text):
    """
    Create a few-shot prompt for Flan-T5 to generate concise summaries that extract actual content.
    
    Args:
        reviews_text: The actual reviews to summarize
    
    Returns:
        Formatted prompt with few-shot examples
    """
    
    # Clean the input text first
    reviews_text = clean_review_text(reviews_text)
    
    # Few-shot examples that demonstrate extracting actual content from reviews
    few_shot_examples = """Summarize the following product reviews in approximately 50 words. Extract key points that customers actually mention: quality, usability, problems, and overall satisfaction.

Reviews: "great tablet for kids. easy to use and durable. my children love it. good parental controls. battery life is excellent. screen is bright and clear."
Customers highlight strong durability and easy child-friendly navigation. Parents appreciate robust parental controls and excellent battery life. The bright, clear screen contributes to high satisfaction. Children enjoy using it, making it a trusted family device.

Reviews: "the kindle is perfect for reading. lightweight and comfortable to hold. battery lasts weeks. screen is easy on the eyes. love the built-in light."
Readers emphasize the lightweight, comfortable design ideal for extended reading sessions. Multi-week battery life is frequently praised. The eye-friendly screen with integrated lighting enhances comfort. Overall highly satisfactory for avid readers seeking convenience.

Reviews: "echo works well but setup was confusing. sound quality is good. alexa responds quickly. had issues connecting to wifi initially. overall satisfied with purchase."
Users report good sound quality and responsive Alexa performance. However, initial setup proved confusing for some, with WiFi connectivity challenges noted. Despite early frustrations, overall satisfaction is positive once operational.

Reviews: {reviews}

Summary:"""
    
    return few_shot_examples.format(reviews=reviews_text)

In [16]:
def summarize_product_reviews(reviews_text, target_words=50, max_input_length=512):
    """
    Summarize product reviews using LoRA fine-tuned Flan-T5 Large to generate content-based summaries.
    
    Args:
        reviews_text: Combined review text for a product
        target_words: Target number of words for summary (default: 50)
        max_input_length: Maximum tokens for input
    
    Returns:
        Summary text extracted from actual review content
    """
    # Create the few-shot prompt (cleaning happens inside)
    prompt = create_few_shot_prompt(reviews_text[:3000])
    
    # Calculate approximate token count for target words
    max_output_length = int(target_words * 1.5)
    min_output_length = int(target_words * 1.2)
    
    # Tokenize input
    inputs = tokenizer(
        prompt,
        max_length=max_input_length,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(device)
    
    # Generate summary with adjusted parameters to encourage content extraction
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_output_length,
            min_length=min_output_length,
            num_beams=5,  # Increased for better quality
            length_penalty=0.8,  # Slightly favor shorter, more focused summaries
            early_stopping=True,
            no_repeat_ngram_size=3,
            temperature=0.9,  # Increased for more diversity
            do_sample=True,  # Enable sampling for more natural text
            top_p=0.92,  # Nucleus sampling
            repetition_penalty=1.2  # Discourage repetition
        )
    
    # Decode and clean summary
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    # Remove any "Summary:" prefix that might appear
    if summary.lower().startswith('summary:'):
        summary = summary[8:].strip()
    
    return summary

### 7.2 Test the Few-Shot Summarization

Test the prompt engineering approach on a sample product before processing all products.

In [17]:
# Test on a sample product
test_product = product_reviews.iloc[0]
test_reviews = test_product['combined_reviews']

print(f"Testing on Product: {test_product['name']}")
print(f"Average Rating: {test_product['avg_rating']:.2f}")
print(f"\nOriginal Reviews (first 500 chars):\n{test_reviews[:500]}...")
print("\n" + "="*80)
print("Generating summary with few-shot prompting...\n")

test_summary = summarize_product_reviews(test_reviews)
print(f"Generated Summary:\n{test_summary}")

Testing on Product: Amazon Fire Tv
Average Rating: 4.71

Original Reviews (first 500 chars):
The Amazon Fire TV works better and faster than the Fire Stick. Every aspect of this is better, watching every show or movie starts up fast and is much clearer. I like the product and would tell others about it. Really like the voice commands, including commands outside the typical, movie/tv expectation (like the weather). Haven't used it as much as i thought i would, but set up (for the things i want to do) is fair.... Got it so I can have everything in one device and it does that flawless. It'...

Generating summary with few-shot prompting...

Generated Summary:
Customers praise the quality of the device. Overall, customers are satisfied with the product. Rating: 4.5/5. Reviewers also praise the voice search feature. Customer ratings: 4.7/5. Reviews: 4.8/5. Ratings: 4.6/5. Customer reviews: 4.3/5.
Generated Summary:
Customers praise the quality of the device. Overall, customers are satisfied 

### 7.3 Generate Summaries for All Products in All Clusters

Process all products and generate summaries using the few-shot approach.

In [18]:
from tqdm import tqdm
import time

# Filter to get only top 3 and worst product from each cluster
products_to_summarize = []

for cluster in sorted(product_reviews_with_counts['cluster'].unique()):
    cluster_data = product_reviews_with_counts[product_reviews_with_counts['cluster'] == cluster]
    
    # Get top 3 products (best ratings/popularity)
    top_3 = cluster_data.nsmallest(3, 'popularity_rank_in_cluster')
    
    # Get worst product (lowest rating)
    worst_1 = cluster_data.nsmallest(1, 'avg_rating')
    
    # Combine and remove duplicates
    cluster_selection = pd.concat([top_3, worst_1]).drop_duplicates(subset=['product_id'])
    products_to_summarize.append(cluster_selection)

# Combine all selected products
products_to_summarize = pd.concat(products_to_summarize).reset_index(drop=True)

print(f"Generating summaries for {len(products_to_summarize)} selected products (top 3 + worst from each cluster)...")
print(f"Clusters: {sorted(products_to_summarize['cluster'].unique())}")
print("Using few-shot prompt engineering with Flan-T5")
print("This may take several minutes...\n")

# Create a list to store all summaries
all_summaries = []

# Process each selected product with progress tracking
for idx, row in tqdm(products_to_summarize.iterrows(), total=len(products_to_summarize), desc="Summarizing"):
    try:
        summary = summarize_product_reviews(row['combined_reviews'])
        
        all_summaries.append({
            'name': row['name'],
            'cluster': row['cluster'],
            'avg_rating': row['avg_rating'],
            'review_count': row['review_count'],
            'popularity_rank': row['popularity_rank_in_cluster'],
            'popularity_score': row['popularity_score'],
            'predicted_sentiment_SVM': row['predicted_sentiment_SVM'],
            'summary': summary
        })
        
        # Small delay to avoid overloading
        time.sleep(0.1)
        
    except Exception as e:
        print(f"\nError processing product {row['name']}: {str(e)}")
        all_summaries.append({
            'name': row['name'],
            'cluster': row['cluster'],
            'avg_rating': row['avg_rating'],
            'review_count': row['review_count'],
            'popularity_rank': row['popularity_rank_in_cluster'],
            'popularity_score': row['popularity_score'],
            'predicted_sentiment_SVM': row['predicted_sentiment_SVM'],
            'summary': f"Error generating summary: {str(e)}"
        })

# Convert to DataFrame
summaries_df = pd.DataFrame(all_summaries)

# Sort by cluster and popularity rank for better organization
summaries_df = summaries_df.sort_values(['cluster', 'popularity_rank']).reset_index(drop=True)

print(f"\n✓ Summarization complete!")
print(f"Total products summarized: {len(summaries_df)}")
print(f"Products per cluster:")
for cluster in sorted(summaries_df['cluster'].unique()):
    count = len(summaries_df[summaries_df['cluster'] == cluster])
    print(f"  Cluster {cluster}: {count} products")
summaries_df.head()

Generating summaries for 14 selected products (top 3 + worst from each cluster)...
Clusters: [np.int64(0), np.int64(1), np.int64(3), np.int64(4)]
Using few-shot prompt engineering with Flan-T5
This may take several minutes...



Summarizing:   0%|          | 0/14 [00:00<?, ?it/s]

Summarizing: 100%|██████████| 14/14 [01:24<00:00,  6.06s/it]


✓ Summarization complete!
Total products summarized: 14
Products per cluster:
  Cluster 0: 2 products
  Cluster 1: 4 products
  Cluster 3: 4 products
  Cluster 4: 4 products


,name,cluster,avg_rating,review_count,popularity_rank,popularity_score,predicted_sentiment_SVM,summary
0,Amazon,0,4.875000,8,1,0.665722,positive,Customers praise the quality of this product. ...
1,Amazon 5W USB Official OEM Charger and Power A...,0,4.440299,401,2,0.556602,positive,Customers praise the quality of the product. R...
2,Amazon Kindle Paperwhite - eBook reader - 4 GB...,1,4.772355,3176,1,0.724093,positive,Customers praise the screen. Rating: 4.5/5. Cu...
3,"Kindle Voyage E-reader, 6 High-Resolution Disp...",1,4.729310,580,2,0.641197,positive,Customers praise the screen. Rating: 4.3/5. Re...
4,Echo (White),1,4.700000,10,3,0.617519,positive,"Customers praise the tablet's quality, usabili..."


### 8. Display Summaries by Cluster

Show sample summaries organized by cluster.

In [19]:
# Display summaries organized by cluster
print("Product Summaries by Cluster (Top 3 + Worst):\n")
print("="*100)

for cluster in sorted(summaries_df['cluster'].unique()):
    cluster_products = summaries_df[summaries_df['cluster'] == cluster].sort_values('popularity_rank')
    
    print(f"\n{'='*100}")
    print(f"CLUSTER {cluster} - {len(cluster_products)} products (Top 3 Best + Worst)")
    print(f"{'='*100}\n")
    
    # Show all products from this cluster
    for idx, (_, product) in enumerate(cluster_products.iterrows(), 1):
        rank_label = f"Rank #{int(product['popularity_rank'])}" if product['popularity_rank'] <= 3 else "WORST"
        print(f"Product {idx} ({rank_label}): {product['name']}")
        print(f"  Rating: {product['avg_rating']:.2f} | Reviews: {int(product['review_count'])} | Sentiment: {product['predicted_sentiment_SVM']}")
        print(f"  Popularity Score: {product['popularity_score']:.3f}")
        print(f"  Summary: {product['summary']}")
        print("-" * 100)

Product Summaries by Cluster (Top 3 + Worst):


CLUSTER 0 - 2 products (Top 3 Best + Worst)

Product 1 (Rank #1): Amazon
  Rating: 4.88 | Reviews: 8 | Sentiment: positive
  Popularity Score: 0.666
  Summary: Customers praise the quality of this product. Rating: 4.5/5. Reviewers also praise the convenience of using this cable with the kindle fire hd. Overall, customers are satisfied with the product and recommend it to friends and family. Reviews: 4.3/5. Customer rating: 4.7/5.
----------------------------------------------------------------------------------------------------
Product 2 (Rank #2): Amazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders
  Rating: 4.44 | Reviews: 401 | Sentiment: positive
  Popularity Score: 0.557
  Summary: Customers praise the quality of the product. Rating: 4.6/5. Customer satisfaction: 4.7/5. Reviewers praise the product's durability and usability. Overall, customers are satisfied with the product and recommend it to 

### 9. Export Summaries to Output File

Save all product summaries to a comprehensive output file.

In [20]:
# Save summaries to CSV file
csv_output_file = 'results/product_summaries.csv'
summaries_df.to_csv(csv_output_file, index=False)
print(f"✓ CSV file saved: {csv_output_file}")

# Also create a detailed text report
text_output_file = 'results/product_summaries_report.txt'

with open(text_output_file, 'w', encoding='utf-8') as f:
    f.write("="*100 + "\n")
    f.write("PRODUCT REVIEW SUMMARIES - FLAN-T5-LARGE with LoRA (50-WORD SUMMARIES)\n")
    f.write("Top 3 Best + Worst Product from Each Cluster\n")
    f.write("="*100 + "\n\n")
    f.write(f"Total Products: {len(summaries_df)}\n")
    f.write(f"Selection: Top 3 best (by popularity) + 1 worst (by rating) from each cluster\n")
    f.write(f"Generation Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model: google/flan-t5-large (~780M params) with LoRA on {device}\n")
    f.write(f"Target Summary Length: 50 words\n")
    f.write("="*100 + "\n\n")
    
    # Group by cluster
    for cluster in sorted(summaries_df['cluster'].unique()):
        cluster_products = summaries_df[summaries_df['cluster'] == cluster].sort_values('popularity_rank')
        
        f.write("\n" + "="*100 + "\n")
        f.write(f"CLUSTER {cluster} ({len(cluster_products)} products: Top 3 Best + Worst)\n")
        f.write("="*100 + "\n\n")
        
        for idx, (_, product) in enumerate(cluster_products.iterrows(), 1):
            rank_label = f"RANK #{int(product['popularity_rank'])} (TOP)" if product['popularity_rank'] <= 3 else "WORST RATED"
            f.write(f"\n{'-'*100}\n")
            f.write(f"Product #{idx} - {rank_label}\n")
            f.write(f"{'-'*100}\n")
            f.write(f"Product Name: {product['name']}\n")
            f.write(f"Cluster: {product['cluster']}\n")
            f.write(f"Average Rating: {product['avg_rating']:.2f} stars\n")
            f.write(f"Review Count: {int(product['review_count'])} reviews\n")
            f.write(f"Popularity Rank in Cluster: #{int(product['popularity_rank'])}\n")
            f.write(f"Popularity Score: {product['popularity_score']:.3f}\n")
            f.write(f"Predicted Sentiment: {product['predicted_sentiment_SVM']}\n")
            f.write(f"\nSUMMARY:\n{product['summary']}\n")
        
        f.write("\n")
    
    f.write("\n" + "="*100 + "\n")
    f.write("END OF REPORT\n")
    f.write("="*100 + "\n")

print(f"✓ Text report saved: {text_output_file}")
print(f"\nSummary Statistics:")
print(f"  - Total products: {len(summaries_df)}")
print(f"  - Products per cluster:")
for cluster in sorted(summaries_df['cluster'].unique()):
    count = len(summaries_df[summaries_df['cluster'] == cluster])
    print(f"    Cluster {cluster}: {count} products")

✓ CSV file saved: results/product_summaries.csv
✓ Text report saved: results/product_summaries_report.txt

Summary Statistics:
  - Total products: 14
  - Products per cluster:
    Cluster 0: 2 products
    Cluster 1: 4 products
    Cluster 3: 4 products
    Cluster 4: 4 products
